# Kickstarter Row-to-Text NLP Embeddings
### Word2Vec · Sentence-Transformers · DistilBERT · BERT

This notebook converts each Kickstarter project's **structured/tabular row** into a **serialized text representation**
(since there is no raw `name`/`blurb` text column available — only derived counts), and then generates
sentence-level embeddings using four different NLP methods:

1. **Word2Vec** (trained on the corpus of serialized rows, mean-pooled)
2. **Sentence-Transformers** (`all-MiniLM-L6-v2`)
3. **DistilBERT** (`distilbert-base-uncased`, mean-pooled last hidden state)
4. **BERT** (`bert-base-uncased`, mean-pooled last hidden state / CLS token)

Each embedding set is saved separately (with `id` as the join key) so they can later be merged
back onto `ML_train.csv` / `ML_test.csv` for downstream regression modeling.

**Input:** `kickstarter_raw_before_encoding.csv`
**Output:** `word2vec_embeddings.csv`, `sbert_embeddings.csv`, `distilbert_embeddings.csv`, `bert_embeddings.csv`


## 1. Install & Import Dependencies

In [ ]:
# Install required libraries (uncomment if running fresh on Kaggle)
# !pip install -q gensim sentence-transformers transformers torch --upgrade


In [1]:
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings("ignore")

import torch
from tqdm.auto import tqdm

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cuda


## 2. Load Raw Dataset

In [2]:
DATA_PATH = r"E:\NSU\cse445\EDA attempt3\Dataset\kickstarter_raw_before_encoding.csv"  # <-- update path as needed
# Fallback for local/project testing:
import os
if not os.path.exists(DATA_PATH):
    DATA_PATH = "kickstarter_raw_before_encoding.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


(20000, 25)


,id,goal_usd,log_goal_usd,duration_days,prelaunch_days,name_char_length,name_word_count,blurb_char_length,blurb_word_count,launch_year,...,prelaunch_activated,country,currency,category_parent,category_name,location_type,location_country,location_state,target_usd,log_target
0,800050031,6000.000000,8.699681,29.958333,8.745764,33,4,128,18,2013,...,0,US,USD,Music,Latin,Town,US,KY,10043.000000,9.214731
1,746075806,11655.899250,9.363653,30.416447,116.766181,12,1,104,19,2018,...,0,CA,CAD,Film & Video,Documentary,Town,CA,ON,36295.918552,10.499488
2,1638770732,106.553578,4.677989,27.000000,4.452442,50,7,104,22,2022,...,1,DE,EUR,Art,Painting,LocalAdmin,DE,Rhineland-Palatinate,3325.537169,8.109687
3,511984273,20000.000000,9.903538,30.000000,14.753009,13,3,62,8,2023,...,1,US,USD,Film & Video,Horror,Town,US,AL,30020.660000,10.309674
4,752979518,477.029064,6.169672,30.603519,27.764248,44,8,99,14,2022,...,0,CA,CAD,Fashion,Accessories,Town,CA,ON,736.445419,6.603192


## 3. Row Serialization (Structured → Text)

Since the dataset has no raw free-text columns, we serialize each row into human-readable
sentence form. This mirrors the "human-readable style" serialization approach used for
NLP/LLM-based structured-data prediction.


In [3]:
def serialize_row(row):
    """Convert one tabular row into a natural-language description."""
    text = (
        f"This Kickstarter project is in the {row['category_parent']} category, "
        f"specifically {row['category_name']}. "
        f"It was launched from {row['location_country']}"
        + (f", {row['location_state']}" if pd.notna(row.get('location_state')) else "")
        + f", with a location type of {row['location_type']}. "
        f"The campaign country is {row['country']} and currency used is {row['currency']}. "
        f"The funding goal was {row['goal_usd']:.2f} USD "
        f"(target amount {row['target_usd']:.2f} USD). "
        f"The campaign ran for {row['duration_days']:.1f} days "
        f"with a prelaunch period of {row['prelaunch_days']:.1f} days. "
        f"The project title had {int(row['name_word_count'])} words "
        f"and the blurb had {int(row['blurb_word_count'])} words. "
        f"It was launched in {int(row['launch_month'])}/{int(row['launch_year'])} "
        f"at hour {int(row['launch_hour'])} on weekday {int(row['launch_weekday'])}. "
        f"{'A video was included.' if row['has_video'] == 1 else 'No video was included.'} "
        f"{'Prelaunch marketing was activated.' if row['prelaunch_activated'] == 1 else 'No prelaunch marketing was activated.'}"
    )
    return text

df["serialized_text"] = df.apply(serialize_row, axis=1)
print(df["serialized_text"].iloc[0])


This Kickstarter project is in the Music category, specifically Latin. It was launched from US, KY, with a location type of Town. The campaign country is US and currency used is USD. The funding goal was 6000.00 USD (target amount 10043.00 USD). The campaign ran for 30.0 days with a prelaunch period of 8.7 days. The project title had 4 words and the blurb had 18 words. It was launched in 2/2013 at hour 12 on weekday 4. A video was included. No prelaunch marketing was activated.


In [4]:
def basic_clean(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["serialized_text"].apply(basic_clean)
corpus = df["clean_text"].tolist()
ids = df["id"].values
print(f"Total documents: {len(corpus)}")


Total documents: 20000


## TF iDF


In [6]:
# ============================================
# TF-IDF Embeddings
# ============================================

from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF configuration
TFIDF_MAX_FEATURES = 5000

tfidf_vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit and transform the same corpus used by the other NLP embeddings
tfidf_embeddings = tfidf_vectorizer.fit_transform(corpus)

print("TF-IDF embeddings shape:", tfidf_embeddings.shape)
print("Number of TF-IDF features:", len(tfidf_vectorizer.get_feature_names_out()))

TF-IDF embeddings shape: (20000, 5000)
Number of TF-IDF features: 5000


In [7]:
# ============================================
# Save TF-IDF Embeddings
# ============================================

tfidf_cols = [
    f"tfidf_{i}" 
    for i in range(tfidf_embeddings.shape[1])
]

tfidf_df = pd.DataFrame(
    tfidf_embeddings.toarray(),
    columns=tfidf_cols
)

tfidf_df.insert(0, "id", ids)

tfidf_df.to_csv(
    "tfidf_embeddings.csv",
    index=False
)

print("Saved: tfidf_embeddings.csv")
print("Shape:", tfidf_df.shape)

tfidf_df.head()

Saved: tfidf_embeddings.csv
Shape: (20000, 5001)


,id,tfidf_0,tfidf_1,tfidf_2,tfidf_3,tfidf_4,tfidf_5,tfidf_6,tfidf_7,tfidf_8,...,tfidf_4990,tfidf_4991,tfidf_4992,tfidf_4993,tfidf_4994,tfidf_4995,tfidf_4996,tfidf_4997,tfidf_4998,tfidf_4999
0,800050031,0.096092,0.096092,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,746075806,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1638770732,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,511984273,0.059373,0.059373,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,752979518,0.000000,0.000000,0.0,0.0,0.0,0.0,0.202698,0.202698,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Word2Vec Embeddings

We train a Word2Vec model directly on the serialized-text corpus, then represent each
document as the **mean of its token vectors**.


In [14]:
from gensim.models import Word2Vec

W2V_DIM = 100

tokenized_corpus = [doc.split() for doc in corpus]

w2v_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=W2V_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=1,          # skip-gram
    epochs=20,
    seed=RANDOM_STATE,
)

print("Word2Vec vocabulary size:", len(w2v_model.wv))


Word2Vec vocabulary size: 25087


In [15]:
def document_vector_w2v(tokens, model, dim):
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if len(vectors) == 0:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

w2v_embeddings = np.vstack([
    document_vector_w2v(tokens, w2v_model, W2V_DIM) for tokens in tqdm(tokenized_corpus, desc="Word2Vec embedding")
])

print("Word2Vec embeddings shape:", w2v_embeddings.shape)


Word2Vec embedding:   0%|          | 0/20000 [00:00<?, ?it/s]

Word2Vec embeddings shape: (20000, 100)


In [16]:
w2v_cols = [f"w2v_{i}" for i in range(W2V_DIM)]
w2v_df = pd.DataFrame(w2v_embeddings, columns=w2v_cols)
w2v_df.insert(0, "id", ids)
w2v_df.to_csv("word2vec_embeddings.csv", index=False)
w2v_df.head()


,id,w2v_0,w2v_1,w2v_2,w2v_3,w2v_4,w2v_5,w2v_6,w2v_7,w2v_8,...,w2v_90,w2v_91,w2v_92,w2v_93,w2v_94,w2v_95,w2v_96,w2v_97,w2v_98,w2v_99
0,800050031,0.024111,0.381261,-0.195732,-0.214749,-0.316735,0.151688,-0.181561,0.500828,-0.095338,...,-0.267207,0.223505,0.044580,-0.040207,0.224435,-0.133920,0.002096,0.162076,-0.301359,-0.288855
1,746075806,0.022816,0.357612,-0.213712,-0.265010,-0.350555,0.153642,-0.185045,0.489420,-0.107655,...,-0.280990,0.209344,0.093290,-0.009761,0.205130,-0.138626,-0.027888,0.166182,-0.330758,-0.288455
2,1638770732,0.047388,0.362403,-0.221607,-0.254770,-0.352191,0.102441,-0.154795,0.454180,-0.101984,...,-0.282321,0.213168,0.055026,-0.037602,0.202478,-0.166214,-0.036278,0.174760,-0.256964,-0.223689
3,511984273,0.026172,0.373360,-0.183256,-0.220617,-0.313346,0.141345,-0.184397,0.483516,-0.096237,...,-0.280984,0.229600,0.062126,-0.060362,0.224474,-0.127719,0.000959,0.178789,-0.322159,-0.295098
4,752979518,0.034207,0.373439,-0.216176,-0.257320,-0.357521,0.164213,-0.178209,0.503174,-0.114648,...,-0.280438,0.191496,0.087183,0.004624,0.206834,-0.152141,-0.031654,0.150412,-0.324935,-0.300680


## 5. Sentence-Transformers Embeddings

Using a pretrained sentence-embedding model (`all-MiniLM-L6-v2`) — fast, strong semantic
quality, 384-dimensional output.


In [17]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

sbert_embeddings = sbert_model.encode(
    corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print("Sentence-Transformers embeddings shape:", sbert_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Sentence-Transformers embeddings shape: (20000, 384)


In [18]:
sbert_cols = [f"sbert_{i}" for i in range(sbert_embeddings.shape[1])]
sbert_df = pd.DataFrame(sbert_embeddings, columns=sbert_cols)
sbert_df.insert(0, "id", ids)
sbert_df.to_csv("sbert_embeddings.csv", index=False)
sbert_df.head()


,id,sbert_0,sbert_1,sbert_2,sbert_3,sbert_4,sbert_5,sbert_6,sbert_7,sbert_8,...,sbert_374,sbert_375,sbert_376,sbert_377,sbert_378,sbert_379,sbert_380,sbert_381,sbert_382,sbert_383
0,800050031,0.009696,-0.022843,-0.005649,-0.044456,0.034605,0.026435,-0.058152,0.005665,-0.026143,...,0.107221,0.016896,-0.002761,0.036294,-0.019178,0.001893,0.013292,-0.091713,-0.022584,0.046567
1,746075806,-0.014528,0.016317,-0.010855,-0.040494,0.044259,0.023785,-0.059390,0.045677,-0.012935,...,0.121136,0.042486,-0.004753,0.012023,-0.027562,0.025531,0.038324,-0.090688,-0.001799,0.047123
2,1638770732,0.001657,0.051111,-0.010981,-0.080065,0.056585,0.027084,-0.048842,0.041670,-0.043780,...,0.104580,0.007488,0.007862,0.010823,-0.002156,0.059182,0.033180,-0.067447,-0.020558,0.042201
3,511984273,-0.014096,0.014086,-0.025287,-0.024354,0.055744,0.012978,-0.040686,0.017293,-0.015719,...,0.165446,-0.009274,-0.013775,0.004518,-0.030238,0.044520,0.033465,-0.085164,-0.013003,0.041919
4,752979518,-0.021410,0.039737,0.008426,-0.015229,0.057119,0.016359,-0.049132,0.033249,-0.055557,...,0.079491,0.033743,-0.010182,0.013082,-0.004943,0.018710,0.028883,-0.119158,-0.022343,0.052983


## 6. DistilBERT Embeddings

Mean-pooled last hidden state from `distilbert-base-uncased` (768-dimensional).


In [19]:
from transformers import AutoTokenizer, AutoModel

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    counts = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return summed / counts

@torch.no_grad()
def embed_texts_transformer(texts, tokenizer, model, device, batch_size=32, max_length=128):
    model.eval()
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        pooled = mean_pooling(output, encoded["attention_mask"])
        all_embeddings.append(pooled.cpu().numpy())
    return np.vstack(all_embeddings)


In [20]:
distilbert_name = "distilbert-base-uncased"
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert_name)
distilbert_model = AutoModel.from_pretrained(distilbert_name).to(DEVICE)

distilbert_embeddings = embed_texts_transformer(
    corpus, distilbert_tokenizer, distilbert_model, DEVICE, batch_size=32, max_length=128
)

print("DistilBERT embeddings shape:", distilbert_embeddings.shape)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding batches:   0%|          | 0/625 [00:00<?, ?it/s]

DistilBERT embeddings shape: (20000, 768)


In [21]:
distilbert_cols = [f"distilbert_{i}" for i in range(distilbert_embeddings.shape[1])]
distilbert_df = pd.DataFrame(distilbert_embeddings, columns=distilbert_cols)
distilbert_df.insert(0, "id", ids)
distilbert_df.to_csv("distilbert_embeddings.csv", index=False)
distilbert_df.head()


,id,distilbert_0,distilbert_1,distilbert_2,distilbert_3,distilbert_4,distilbert_5,distilbert_6,distilbert_7,distilbert_8,...,distilbert_758,distilbert_759,distilbert_760,distilbert_761,distilbert_762,distilbert_763,distilbert_764,distilbert_765,distilbert_766,distilbert_767
0,800050031,-0.174869,-0.153884,0.334860,0.231170,0.077124,0.062045,0.133232,0.454254,0.171575,...,0.191802,-0.066824,0.212809,-0.259939,-0.027766,0.034210,-0.027507,-0.157761,0.095013,-0.018019
1,746075806,-0.050175,-0.138276,0.323678,0.181142,0.089607,0.045732,0.067510,0.434548,0.133393,...,0.146245,-0.072120,0.157449,-0.270784,-0.034000,0.070286,-0.062972,-0.194394,0.097627,0.007913
2,1638770732,-0.145467,-0.100403,0.369016,0.100974,0.064012,0.039819,0.097862,0.398956,0.147769,...,0.256543,-0.072497,0.194673,-0.342051,0.014132,0.033948,-0.111386,-0.147802,0.092507,0.015587
3,511984273,-0.105532,-0.151397,0.358972,0.206332,0.083037,0.022751,0.092605,0.419307,0.141954,...,0.137791,-0.103637,0.164830,-0.280794,-0.078928,0.146960,-0.043144,-0.163061,0.142563,-0.055782
4,752979518,-0.134750,-0.152774,0.336347,0.155386,0.108033,0.065137,0.045647,0.463279,0.136027,...,0.159849,-0.063034,0.182155,-0.297007,0.019297,0.014496,-0.080417,-0.205194,0.079773,0.024113


## 7. BERT Embeddings

Mean-pooled last hidden state from `bert-base-uncased` (768-dimensional).


In [22]:
bert_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name).to(DEVICE)

bert_embeddings = embed_texts_transformer(
    corpus, bert_tokenizer, bert_model, DEVICE, batch_size=32, max_length=128
)

print("BERT embeddings shape:", bert_embeddings.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding batches:   0%|          | 0/625 [00:00<?, ?it/s]

BERT embeddings shape: (20000, 768)


In [23]:
bert_cols = [f"bert_{i}" for i in range(bert_embeddings.shape[1])]
bert_df = pd.DataFrame(bert_embeddings, columns=bert_cols)
bert_df.insert(0, "id", ids)
bert_df.to_csv("bert_embeddings.csv", index=False)
bert_df.head()


,id,bert_0,bert_1,bert_2,bert_3,bert_4,bert_5,bert_6,bert_7,bert_8,...,bert_758,bert_759,bert_760,bert_761,bert_762,bert_763,bert_764,bert_765,bert_766,bert_767
0,800050031,-0.131295,-0.201791,0.410148,0.181397,0.009526,0.045876,0.190163,0.402379,0.102221,...,0.244116,-0.101161,0.354765,-0.320509,-0.003776,0.109308,-0.026094,-0.210480,0.078259,0.097913
1,746075806,-0.072396,-0.163510,0.371733,0.147683,0.017034,-0.018327,0.116827,0.407074,0.101593,...,0.212313,-0.145414,0.380709,-0.344923,-0.017827,0.086621,-0.064178,-0.245141,0.117370,0.063120
2,1638770732,-0.241007,-0.112611,0.390516,0.101298,0.096635,-0.064311,0.197366,0.393352,0.103321,...,0.324022,-0.056320,0.384191,-0.340616,-0.017965,0.044519,-0.165939,-0.283587,0.119357,0.022223
3,511984273,-0.073712,-0.232313,0.435276,0.160809,0.032171,-0.047701,0.189497,0.407740,0.124027,...,0.174748,-0.122199,0.358033,-0.332812,-0.030904,0.157724,-0.024459,-0.266788,0.144655,-0.003961
4,752979518,-0.120321,-0.219804,0.358538,0.132307,0.059496,0.001448,0.090509,0.436796,0.072462,...,0.243335,-0.106613,0.382058,-0.340044,0.060117,0.053553,-0.060467,-0.298618,0.104075,0.030117


## 8. Summary

| Method | Dim | Output file |
|---|---|---|
| Word2Vec (trained, mean-pooled) | 100 | `word2vec_embeddings.csv` |
| Sentence-Transformers (`all-MiniLM-L6-v2`) | 384 | `sbert_embeddings.csv` |
| DistilBERT (`distilbert-base-uncased`, mean-pooled) | 768 | `distilbert_embeddings.csv` |
| BERT (`bert-base-uncased`, mean-pooled) | 768 | `bert_embeddings.csv` |

**Next steps (not part of this notebook):**
- Merge any of these embedding sets onto `ML_train.csv` / `ML_test.csv` via the `id` column.
- Optionally reduce dimensionality (e.g. PCA) before merging, to avoid these high-dimensional
  embeddings dominating the existing ~80 tabular features.
- Feed the merged feature set into your existing 9-model × 3-tuning-method pipeline, and compare
  performance with vs. without the NLP embedding features.


In [24]:
print("All embeddings generated and saved:")
for fname in ["word2vec_embeddings.csv", "sbert_embeddings.csv", "distilbert_embeddings.csv", "bert_embeddings.csv"]:
    import os
    if os.path.exists(fname):
        print(f" - {fname}: {pd.read_csv(fname).shape}")


All embeddings generated and saved:
 - word2vec_embeddings.csv: (20000, 101)
 - sbert_embeddings.csv: (20000, 385)
 - distilbert_embeddings.csv: (20000, 769)
 - bert_embeddings.csv: (20000, 769)
